# DexGrasp training on Kaggle

Before running, enable a GPU accelerator and Internet access in the Kaggle notebook settings. Add `WANDB_API_KEY` to Kaggle Secrets. `WANDB_ENTITY` is optional and is only needed when the target project belongs to a team.

The smoke run should pass before starting the full run. Both paths enable NaNGuard, upload checkpoints, preserve the exact config and git state, and attach a diagnostics artifact to the same W&B run even when training exits with an error.

In [ ]:
import json
import os
import secrets
import shlex
import subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/hunghehe2205/mjlab.git"
BRANCH = "DexGrasp"
REPO_DIR = Path("/kaggle/working/mjlab")
UV = "/kaggle/working/bin/uv"
LOG_ROOT = Path("/kaggle/working/mjlab_logs")
DIAGNOSTICS_ROOT = Path("/kaggle/working/mjlab_diagnostics")
WANDB_PROJECT = "mjlab-dexgrasp"
TASK = "Mjlab-DexGrasp-UR5eRH5DG2"

os.environ.update(
  {
    "MJLAB_REPO_URL": REPO_URL,
    "MJLAB_BRANCH": BRANCH,
    "MJLAB_REPO_DIR": str(REPO_DIR),
    "MUJOCO_GL": "egl",
    "PYTHONUNBUFFERED": "1",
  }
)

## Clone and switch branch

In [ ]:
%%bash
set -euo pipefail
if [ ! -d "$MJLAB_REPO_DIR/.git" ]; then
  git clone --branch "$MJLAB_BRANCH" --single-branch "$MJLAB_REPO_URL" "$MJLAB_REPO_DIR"
else
  git -C "$MJLAB_REPO_DIR" fetch origin "$MJLAB_BRANCH"
  git -C "$MJLAB_REPO_DIR" switch "$MJLAB_BRANCH"
  git -C "$MJLAB_REPO_DIR" pull --ff-only origin "$MJLAB_BRANCH"
fi
git -C "$MJLAB_REPO_DIR" status --short --branch
git -C "$MJLAB_REPO_DIR" log -1 --oneline

## Install the locked CUDA environment

In [ ]:
%%bash
set -euo pipefail
mkdir -p /kaggle/working/bin
if [ ! -x /kaggle/working/bin/uv ]; then
  export UV_INSTALL_DIR=/kaggle/working/bin
  curl -LsSf https://astral.sh/uv/install.sh | sh
fi
cd "$MJLAB_REPO_DIR"
/kaggle/working/bin/uv sync --frozen --no-dev --extra cu128
/kaggle/working/bin/uv run --extra cu128 python -c "import torch; import warp as wp; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available()); print('warp', wp.__version__)"
nvidia-smi

## Authenticate W&B from Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()
wandb_key = secret_client.get_secret("WANDB_API_KEY")
try:
  wandb_entity = secret_client.get_secret("WANDB_ENTITY")
except Exception:
  wandb_entity = ""

os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_CONSOLE"] = "wrap"
if wandb_entity:
  os.environ["WANDB_ENTITY"] = wandb_entity
  os.environ["WANDB_USERNAME"] = wandb_entity

login_script = (
  "import os, wandb; wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)"
)
subprocess.run(
  [UV, "run", "--extra", "cu128", "python", "-c", login_script],
  cwd=REPO_DIR,
  check=True,
  stdout=subprocess.DEVNULL,
)
print(f"W&B login ready; project={WANDB_PROJECT}, entity={wandb_entity or 'default'}")

## Training and diagnostics helpers

Each run gets a known W&B run ID. The finalizer resumes that same run and uploads run metadata, resolved YAML configs, git diff, and any NaNGuard dumps.

In [ ]:
UPLOAD_SCRIPT = r"""
import json
import os
from pathlib import Path
import wandb

run = wandb.init(
    project=os.environ["UPLOAD_PROJECT"],
    entity=os.environ.get("UPLOAD_ENTITY") or None,
    id=os.environ["UPLOAD_RUN_ID"],
    resume="allow",
)
artifact = wandb.Artifact(
    f"dexgrasp-diagnostics-{run.id}",
    type="debug",
    description="Kaggle run metadata, resolved configs, git state, and NaNGuard dumps.",
)
diag_dir = Path(os.environ["UPLOAD_DIAG_DIR"])
artifact.add_file(str(diag_dir / "run_metadata.json"), name="run_metadata.json")
log_dir_text = os.environ.get("UPLOAD_LOG_DIR", "")
if log_dir_text:
    log_dir = Path(log_dir_text)
    for relative in ("params/env.yaml", "params/agent.yaml", "git/mjlab.diff"):
        path = log_dir / relative
        if path.is_file():
            artifact.add_file(str(path), name=relative)
nan_dir = diag_dir / "nan_dumps"
if nan_dir.is_dir() and any(path.is_file() for path in nan_dir.iterdir()):
    artifact.add_dir(str(nan_dir), name="nan_dumps")
evaluation_text = os.environ.get("UPLOAD_EVALUATION", "")
if evaluation_text and Path(evaluation_text).is_file():
    evaluation_path = Path(evaluation_text)
    artifact.add_file(str(evaluation_path), name="evaluation/lift_success.json")
    metrics = json.loads(evaluation_path.read_text())
    for name, value in metrics.items():
        run.summary[f"eval/lift_success/{name}"] = value
    run.summary["eval/lift_success_mean"] = sum(metrics.values()) / len(metrics)
metadata = json.loads((diag_dir / "run_metadata.json").read_text())
run.summary["code/commit"] = metadata["commit"]
run.summary["code/branch"] = metadata["branch"]
run.summary["debug/train_returncode"] = metadata["returncode"]
run.log_artifact(artifact)
run.finish()
"""


def _latest_log_dir(run_name):
  experiment_dir = LOG_ROOT / "dexgrasp_teacher_ur5e_rh5dg2"
  candidates = sorted(experiment_dir.glob(f"*_{run_name}"))
  return candidates[-1] if candidates else None


def upload_diagnostics(info, evaluation_file=None):
  upload_env = os.environ.copy()
  upload_env.update(
    {
      "UPLOAD_PROJECT": WANDB_PROJECT,
      "UPLOAD_ENTITY": os.environ.get("WANDB_ENTITY", ""),
      "UPLOAD_RUN_ID": info["run_id"],
      "UPLOAD_DIAG_DIR": info["diagnostics_dir"],
      "UPLOAD_LOG_DIR": info.get("log_dir") or "",
      "UPLOAD_EVALUATION": str(evaluation_file or ""),
    }
  )
  subprocess.run(
    [UV, "run", "--extra", "cu128", "python", "-c", UPLOAD_SCRIPT],
    cwd=REPO_DIR,
    env=upload_env,
    check=True,
  )


def run_training(
  run_name, num_envs, max_iterations, num_steps, save_interval, video=False
):
  run_id = secrets.token_hex(4)
  diagnostics_dir = DIAGNOSTICS_ROOT / run_id
  nan_dir = diagnostics_dir / "nan_dumps"
  nan_dir.mkdir(parents=True, exist_ok=True)
  run_env = os.environ.copy()
  run_env.update(
    {
      "WANDB_RUN_ID": run_id,
      "WANDB_RESUME": "allow",
    }
  )
  tags = str(("kaggle", "dexgrasp", "nan-guard"))
  command = [
    UV,
    "run",
    "--extra",
    "cu128",
    "train",
    TASK,
    "--gpu-ids",
    "[0]",
    "--env.scene.num-envs",
    str(num_envs),
    "--agent.num-steps-per-env",
    str(num_steps),
    "--agent.max-iterations",
    str(max_iterations),
    "--agent.save-interval",
    str(save_interval),
    "--agent.run-name",
    run_name,
    "--agent.logger",
    "wandb",
    "--agent.wandb-project",
    WANDB_PROJECT,
    "--agent.wandb-tags",
    tags,
    "--agent.upload-model",
    "True",
    "--enable-nan-guard",
    "True",
    "--env.sim.nan-guard.output-dir",
    str(nan_dir),
    "--log-root",
    str(LOG_ROOT),
    "--video",
    str(video),
  ]
  if video:
    command += ["--video-length", "200", "--video-interval", "2000"]
  print(shlex.join(command))
  completed = subprocess.run(command, cwd=REPO_DIR, env=run_env)
  log_dir = _latest_log_dir(run_name)
  metadata = {
    "run_id": run_id,
    "run_name": run_name,
    "project": WANDB_PROJECT,
    "branch": BRANCH,
    "commit": subprocess.check_output(
      ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip(),
    "started_from_kaggle": True,
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "returncode": completed.returncode,
    "command": command,
    "log_dir": str(log_dir) if log_dir else None,
    "nan_model_note": (
      "NanGuard MJB stores the host template model; use NPZ as the source "
      "of truth for per-world variant state."
    ),
  }
  (diagnostics_dir / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
  )
  info = {
    **metadata,
    "diagnostics_dir": str(diagnostics_dir),
  }
  upload_diagnostics(info)
  if completed.returncode != 0:
    raise subprocess.CalledProcessError(completed.returncode, command)
  return info

## Smoke run

This reproduces the small validation setup used for the physics containment fix. Confirm that `Episode_Termination/nan` remains zero before launching the full job.

In [ ]:
smoke_info = run_training(
  run_name=f"kaggle_smoke_{datetime.now():%Y%m%d_%H%M%S}",
  num_envs=8,
  max_iterations=20,
  num_steps=8,
  save_interval=10,
  video=False,
)
smoke_info

## Full training

The task default is 88 environments and 10,000 learning iterations. Lower `num_envs` if the Kaggle GPU runs out of memory. Video is optional and adds rendering overhead.

In [ ]:
full_info = run_training(
  run_name=f"kaggle_full_{datetime.now():%Y%m%d_%H%M%S}",
  num_envs=88,
  max_iterations=10_000,
  num_steps=70,
  save_interval=100,
  video=False,
)
full_info

## Optional per-object evaluation

This runs lift evaluation for the whole training cohort, stores each object's success rate in the W&B run summary, and adds the JSON to the diagnostics artifact. Start with 32 environments if Kaggle time is limited.

In [ ]:
log_dir = Path(full_info["log_dir"])
checkpoints = sorted(
  log_dir.glob("model_*.pt"),
  key=lambda path: int(path.stem.split("_")[-1]),
)
checkpoint = checkpoints[-1]
evaluation_file = Path(full_info["diagnostics_dir"]) / "lift_success.json"
subprocess.run(
  [
    UV,
    "run",
    "--extra",
    "cu128",
    "dexgrasp-eval",
    "--checkpoint",
    str(checkpoint),
    "--num-envs",
    "32",
    "--device",
    "cuda:0",
    "--output-file",
    str(evaluation_file),
  ],
  cwd=REPO_DIR,
  check=True,
)
upload_diagnostics(full_info, evaluation_file=evaluation_file)
json.loads(evaluation_file.read_text())